# 📊 Análise Financeira — Banco Master S/A (2020–2024)

**Tema:** Crescimento acelerado e riscos de solvência  
**Fonte:** BCB/IF.data + Demonstrações Financeiras publicadas  
**CNPJ:** 33.923.798/0001-00


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.coleta_dados import carregar_dados_locais, calcular_indicadores

df = carregar_dados_locais()
df = calcular_indicadores(df)
df[['periodo','ativo_total_bi','carteira_credito_bi','patrimonio_liquido_bi',
    'lucro_liquido_mi','indice_basileia_pct','roe_pct']].round(2)


## 1. Estatísticas Descritivas

In [ ]:
df[['ativo_total_bi','carteira_credito_bi','depositos_totais_bi',
    'patrimonio_liquido_bi','lucro_liquido_mi','indice_basileia_pct',
    'roe_pct','alavancagem']].describe().round(2)


## 2. Crescimento do Ativo Total

In [ ]:
import plotly.express as px
fig = px.line(df, x='periodo', y='ativo_total_bi',
              title='Ativo Total — Banco Master (R$ bi)',
              markers=True, template='plotly_white')
fig.show()


## 3. Índice de Basileia vs Mínimo Regulatório

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_hline(y=11, line_dash='dash', line_color='red',
              annotation_text='Mínimo regulatório (11%)')
fig.add_trace(go.Scatter(x=df['periodo'], y=df['indice_basileia_pct'],
                          mode='lines+markers', name='Basileia (%)'))
fig.update_layout(title='Índice de Basileia — Banco Master',
                  template='plotly_white', yaxis=dict(range=[9,17]))
fig.show()


## 4. Predição — Regressão Linear do Ativo Total

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.arange(len(df)).reshape(-1,1)
y = df['ativo_total_bi'].values
model = LinearRegression().fit(X, y)

print(f"R²: {model.score(X, y):.4f}")
print(f"Coeficiente (R$ bi/semestre): {model.coef_[0]:.2f}")

X_proj = np.arange(len(df) + 2).reshape(-1,1)
y_proj = model.predict(X_proj)
periodos = df['periodo'].tolist() + ['2025-S1', '2025-S2']

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['periodo'], y=y.round(1),
                          mode='lines+markers', name='Histórico'))
fig.add_trace(go.Scatter(x=periodos, y=y_proj.round(1),
                          mode='lines+markers', name='Projeção',
                          line=dict(dash='dash', color='red')))
fig.add_vline(x=df['periodo'].iloc[-1], line_dash='dot', line_color='gray')
fig.update_layout(title='Projeção do Ativo Total — 2025', template='plotly_white')
fig.show()


## 5. Conclusões

- O Banco Master cresceu **7,7x** em ativos entre 2020 e 2024
- O Índice de Basileia caiu de ~13,2% → **11,9%** (próximo ao mínimo regulatório de 11%)
- A alavancagem atingiu **13,4x** no 2º sem/2024
- R$ 7,6 bi em CDBs venciam até junho/2025 → risco de liquidez concentrado
- A aquisição pelo BRB em março/2025 foi um movimento de estabilização prudencial
